# ViNumQA few-shot — Qwen3-4B-Thinking-2507 (base, no fine-tuning), via vLLM on Modal

Baseline run for `unsloth/Qwen3-4B-Thinking-2507` at 3 shot(s), to complete the fairness
comparison: the SFT side now fine-tunes this exact model on ViNumQA
(`sft-w-reasoning-trace-distill/qwen3-4b-thinking-2507-stf-w-reasoning-trace.ipynb`), but no
few-shot baseline for it existed yet -- every other model in the results table has one. Without
this, the SFT run's reported gain would implicitly be measured against a *different* model's
starting point, which is not a fair before/after.

Same `SYSTEM_MESSAGE` / `USER_MESSAGE_FRAME` / exemplar(s) as the
rest of the few-shot notebooks in this repo (byte-identical, copied from
`vsf-few-shot-vinumqa-glm5.2.ipynb`),
so the only variable is the model.

**Run on Modal** (vLLM, single GPU -- this is a dense 4B model, not the MoE 80B teacher, so
`tensor_parallel_size=1` and no `moe_backend` override are needed), following the same
Server-notebook pattern as `distill-reasoning-trace/qwen3-next-80b-conr-trace-gen-independent-solve.ipynb`:
- HF cache pointed at the attached Modal Volume so the model only downloads once.
- `test.json` uploaded manually via the Server Web UI, read from `/root/test.json` -- update
  `TEST_JSON_PATH` below if you put it elsewhere.
- `scorer.py` is embedded verbatim in this notebook (same content as
  `notebooks/evaluate/scorer.py`) rather than imported, since Modal has no access to the repo.

**Qwen3-4B-Thinking-2507 is thinking-only**: its chat template always opens a `<think>` block and
`enable_thinking=False` is not supported (per the model card) -- unlike the plain Qwen3-4B baseline
notebook in this folder, which explicitly disables thinking. `max_tokens` is set well above what a
terse baseline answer needs to leave room for the reasoning it cannot skip, and the generated
program is recovered by taking everything after the model's own `</think>` (it does not emit a
literal opening tag -- the template already puts it in thinking mode -- so only the close tag
appears in the decoded text). Sampling uses the model card's recommended `temperature=0.6,
top_p=0.95, top_k=20` for Thinking mode rather than this repo's usual `temperature=0.0` for
baselines: greedy decoding is known to degrade into repetition on Qwen3 thinking models, so forcing
it here for cross-notebook consistency would bias this baseline downward for a reason that has
nothing to do with the model's real capability.

### HF cache (Modal Volume)

In [ ]:
import os
from pathlib import Path

# Point the HuggingFace cache at the attached Modal Volume (mounted at
# /mnt/qwen3-4b-thinking) instead of the container's ephemeral local disk, so the
# model download only ever happens once -- subsequent kernel/container
# restarts load from the Volume instead of re-downloading. Must run before
# `from vllm import LLM, ...` below (env vars are read at import time).
# If your Volume is mounted at a different path, update this to match.
HF_CACHE = Path("/mnt/qwen3-4b-thinking")
os.environ["HF_HUB_CACHE"] = str(HF_CACHE)

if not HF_CACHE.exists():
    print(f"!! {HF_CACHE} does not exist -- no Volume attached at that path.")
    print("   Attach one via the notebook sidebar (filesystem tab), otherwise")
    print("   the model downloads to ephemeral disk and is lost on shutdown.")
else:
    hits = list(HF_CACHE.glob("**/models--unsloth--Qwen3-4B-Thinking-2507"))
    if not hits:
        hits = list(HF_CACHE.glob("**/models--Qwen--Qwen3-4B-Thinking-2507"))
    if not hits:
        print(f"{HF_CACHE} is attached but the model is NOT cached yet -- this run will download it.")
    else:
        size_gb = sum(f.stat().st_size for f in hits[0].rglob("*") if f.is_file()) / 1024**3
        print(f"Model already cached in the Volume: {hits[0]}")
        print(f"   cached size: {size_gb:.1f} GB -> no re-download expected.")


In [ ]:
%%capture
!pip install -q vllm tabulate
!pip install -q -U "typing_extensions>=4.12" "pydantic>=2.9"
# IMPORTANT: restart the kernel after this cell finishes (Kernel > Restart),
# then re-run from the top -- same reason as in the 80B teacher notebook: the
# server's base image ships an older typing_extensions already imported into
# the running process, and upgrading the package on disk doesn't unload it.


### Load test.json

In [ ]:
import json
from pathlib import Path
import pandas as pd
from tabulate import tabulate

# Uploaded manually via the Server Web UI file browser -- adjust if you put
# it somewhere else.
TEST_JSON_PATH = Path("/root/test.json")
OUTPUT_DIR = Path("/mnt/qwen3-4b-thinking/outputs/baseline_qwen3-4b-thinking-2507")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TEST_JSON_PATH.exists():
    raise FileNotFoundError(
        f"{TEST_JSON_PATH} not found. Upload test.json via the Server Web UI "
        "file browser first, or update TEST_JSON_PATH to match where you put it."
    )

df = pd.read_json(TEST_JSON_PATH)
print(f"Loaded {len(df)} samples from {TEST_JSON_PATH}")


In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
df["table_processed"] = df.apply(formatting_table, axis=1)
df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
df["input_question"] = df.apply(processing_input_question, axis=1)
df["program_processed"] = df.apply(processing_program_content, axis=1)
df["answer_processed"] = df.apply(processing_answer_content, axis=1)

df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed",
         "input_question", "program_processed", "answer_processed"]]
df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
df["generated_program"] = ""
df.sample(n=3)


### Prompt (few-shot, same as the rest of the repo)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}
 
[TABLE]
{table}
 
[TEXT AFTER TABLE]
{post_text}
 
### QUESTION:
{question}
 
### PROGRAM:"""

# 3 few-shot demonstrations sampled from train.json, one per evidence type
# (Table Only / Text Only / Table & Text), following the ViNumQA task's own
# categorization (see VLSP 2025 NumQA paper, Section 3.3). Kept identical to
# vsf-few-shot-vinumqa-glm5.2.ipynb so results are comparable across models.
# The "table" field of each shot is pre-rendered to the same GitHub-markdown
# format that formatting_table produces for the real data, so the few-shot
# demonstrations are formatted identically to the actual queries.
FEW_SHOT_EXAMPLES = [
    {
        # Table Only (train idx 1959): answer is derived purely from two table cells.
        "pre_text": "phụ lục iv ace limited và các công ty con thông tin bổ sung về phí tái bảo hiểm thu được cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la mỹ, ngoại trừ tỷ lệ phần trăm) số tiền trực tiếp nhượng cho các công ty nhận từ các công ty khác số tiền ròng tỷ lệ phần trăm số tiền nhận được trên.",
        "table": (
            "|   cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la Mỹ, ngoại trừ tỷ lệ phần trăm) | số tiền trực tiếp   | nhượng cho các công ty khác   | nhận từ các công ty khác   | số tiền ròng   | tỷ lệ phần trăm số tiền nhận được trên số tiền ròng   |\n"
            "|----------------------------------------------------------------------------------------------------------------------|---------------------|-------------------------------|----------------------------|----------------|-------------------------------------------------------|\n"
            "|                                                                                                                 2010 | $ 15780             | $ 5792                        | $ 3516                     | $ 13504        | 26% ( 26 % )                                          |\n"
            "|                                                                                                                 2009 | $ 15415             | $ 5943                        | $ 3768                     | $ 13240        | 28% ( 28 % )                                          |\n"
            "|                                                                                                                 2008 | $ 16087             | $ 6144                        | $ 3260                     | $ 13203        | 25% ( 25 % )                                          |"
        ),
        "post_text": ".",
        "question": "Sự khác biệt giữa số tiền chuyển giao và nhận chuyển giao trong năm 2010 là bao nhiêu?",
        "program": "subtract(5792, 3516)",
    },
    {
        # Text Only (train idx 93): the supplied table (VHM financial summary) is
        # irrelevant to the question; the program only uses numbers from pre_text.
        "pre_text": "hệ số khả năng thanh toán lãi vay cũng tăng cao đạt mức 13.2 lần, so với chỉ 10.3 lần cùng kỳ.",
        "table": (
            "|                   |   FY 2015 |   FY 2016 |   FY 2017 |   FY 2018 |   FY 2019(F) |\n"
            "|-------------------|-----------|-----------|-----------|-----------|--------------|\n"
            "| Doanh thu (VNDbn) |      4920 |     11217 |     15297 |     38664 |        71115 |\n"
            "| Lãi gộp (Vbn)     |       718 |      2420 |      3128 |      7617 |        10983 |"
        ),
        "post_text": ".",
        "question": "Hệ số khả năng thanh toán lãi vay tăng bao nhiêu lần so với cùng kỳ năm ngoái?",
        "program": "subtract(13.2, 10.3)",
    },
    {
        # Table & Text (train idx 1127): must locate the right table row ("Nội dung số")
        # across three columns and chain two operators via the #0 reference.
        "pre_text": "tỷ lệ phần trăm chi phí vốn trên phần trăm tổng tài sản của mảng viễn thông được duy trì trên 1, cho thấy sự tập trung phân bổ chi phí vốn vào mảng viễn thông của fpt qua các năm.\nngoài ra, tỷ lệ này của mảng đầu tư và giáo dục là 1,1 vào năm 2018 và 0,9 vào năm 2019, khẳng định fpt cũng đang tập trung vào phát triển 2 mảng này trong 2 năm gần đây.",
        "table": (
            "|                     |   2014 |   2015 |   2016 |   2017 |   2018 |   2019 |\n"
            "|---------------------|--------|--------|--------|--------|--------|--------|\n"
            "| Viễn thông          |    1.8 |    2.4 |    1.9 |    1.4 |    1.7 |    1.8 |\n"
            "| Nội dung số         |    0.4 |    0.2 |    0.9 |    0.1 |    0.1 |    0.1 |\n"
            "| Phát triển phần mềm |    2.4 |    1.3 |    3.1 |    1.1 |    0.4 |    0.5 |"
        ),
        "post_text": ".",
        "question": "Tổng tỷ lệ của mảng Nội dung số trong ba năm từ 2014 đến 2016 là bao nhiêu?",
        "program": "add(0.4, 0.2), add(#0, 0.9)",
    },
]

### Load Qwen3-4B-Thinking-2507 with vLLM

In [ ]:
import os

# Kept from the 80B teacher notebook as a precaution: this was needed on the
# RTX PRO 6000 (Blackwell, SM120) GPUs previously used on this Modal setup --
# vLLM's default FlashInfer sampler doesn't recognize SM120 as supported yet
# and raises at engine-init time. Harmless on other GPU architectures (just
# forces the plain PyTorch-native top-k/top-p sampler instead). Must be set
# before `from vllm import ...` (env vars are read at import time). Remove if
# you confirm your GPU doesn't need it.
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from vllm import LLM, SamplingParams

MODEL_NAME = "unsloth/Qwen3-4B-Thinking-2507"

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,          # dense 4B model, one GPU is plenty
    dtype="bfloat16",
    max_model_len=16384,             # context (~2-3k) + few-shot prefix + generous <think> budget
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
)

tokenizer = llm.get_tokenizer()
print(f"Loaded {MODEL_NAME} on 1 GPU.")


### Generate (batched via vLLM)

In [ ]:
# Build the fixed few-shot prefix once: alternating user/assistant turns, one
# pair per FEW_SHOT_EXAMPLES entry, each following the same USER_MESSAGE_FRAME
# used for the real query so the demonstrations and the query are formatted
# identically.
few_shot_messages = []
for shot in FEW_SHOT_EXAMPLES:
    few_shot_messages.append({
        "role": "user",
        "content": USER_MESSAGE_FRAME.format(
            pre_text=shot["pre_text"], table=shot["table"],
            post_text=shot["post_text"], question=shot["question"],
        ),
    })
    few_shot_messages.append({"role": "assistant", "content": shot["program"]})

prompts = []
for _, row in df.iterrows():
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        *few_shot_messages,
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)

print(f"Built {len(prompts)} few-shot prompts ({len(FEW_SHOT_EXAMPLES)} demonstrations each).")

# Qwen's own model card recommends temperature=0.6, top_p=0.95, top_k=20 for
# Thinking mode -- greedy decoding (temperature=0, used elsewhere in this repo
# for non-thinking baselines) is known to degrade into repetition on Qwen3
# thinking models, so it is deliberately NOT used here.
# max_tokens=8192: this model cannot skip its <think> block, and an unfine-tuned
# model doing a genuinely novel task can reason at length -- same budget used
# for every other reasoning model in this repo (GLM-5.x, the 80B teacher).
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    max_tokens=8192,
    n=1,
)

outputs = llm.generate(prompts, sampling_params)
print(f"Generated {len(outputs)} responses.")


### Extract the program (strip the `<think>` block)

In [ ]:
# The chat template opens thinking mode implicitly; the model's own decoded
# output therefore contains no literal opening <think> tag, only the closing
# </think> once it commits to an answer -- same behavior documented for
# Qwen3-Next-Thinking in the 80B teacher notebook. Content after the first
# </think> is the program; a response with no </think> ran out of budget
# still reasoning, so it is left empty rather than guessed at.
n_truncated = 0
for (_, row), output in zip(df.iterrows(), outputs):
    raw_text = output.outputs[0].text
    close_idx = raw_text.find("</think>")
    if close_idx == -1:
        df.at[row.name, "generated_program"] = ""
        n_truncated += 1
    else:
        df.at[row.name, "generated_program"] = raw_text[close_idx + len("</think>"):].strip()

print(f"Extracted {len(df)} programs. {n_truncated} responses never closed </think> "
      f"(truncated at max_tokens={sampling_params.max_tokens}) -- left empty, will score 0.")


### Score (embedded scorer.py -- Modal has no repo access)

In [ ]:
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.

The shared-task paper states that "the official evaluation protocol proposed by
Chen et al. (2021) is adopted", so the semantics here follow `evaluate/evaluate.py`
(FinQA's own script) rather than being reinvented:

  * Program Accuracy is *symbolic* equivalence, via sympy, between the gold and
    predicted expressions -- not a string or structural match. A prediction may
    reorder or restructure the arithmetic, but it may only use literals that
    appear in the gold program, so it cannot invent constants such as the `100`
    of a percentage rescaling.
  * Execution Accuracy compares the executed result to `exe_ans` exactly, after
    rounding to 5 decimals. No tolerance.
  * `greater` yields the strings "yes"/"no", matching how the dataset stores
    those answers.
  * Every step takes exactly two arguments. Verified against the data: all 663
    steps across the gold programs are binary, and `table_*` always takes a row
    label plus `none` (454 occurrences) rather than a list of values (2).
  * FinQA's `const_` tokens are still understood.

Five corrections are applied, each because the unmodified script cannot
reproduce ViNumQA's own gold, not because the protocol was thought wrong:

1. Tokenisation of bracketed row labels. `program_tokenization` splits on every
   bracket, so `table_min(ROE (%), none)` shatters into six tokens and fails the
   four-tokens-per-step structure check. 35 of the 497 test programs name a row
   whose label contains brackets -- `ROE (%)`, `EPS (VND)`, `P/E (x)` -- and all
   35 were unscoreable. Tokenisation is now bracket-depth aware.

2. Accounting negatives. Tables write negative amounts as `(3344)`. The original
   `process_row` takes the text before the first bracket, leaving an empty
   string, so the cell fails to parse. The dataset's own `exe_ans` was computed
   with those values -- e.g. `table_min(LN hoạt động (tỷ đồng), none)` expects
   -3344 from a row holding `(3344)`. The `-1046 ( 1046 )` form the original
   handled correctly is unchanged.

3. Unparseable cells no longer void the whole row. Measured over the 393 gold
   `table_*(<row>, none)` programs in train, skipping such cells reproduces
   `exe_ans` for 386 against 381 when the row is voided, so skipping is what the
   dataset was built with.

4. `exe_ans` is stored as a string here ("31.0") where FinQA stores a number, so
   the comparison `exe_res == gold_res` was never true. It is coerced, leaving
   the "yes"/"no" answers alone.

5. The `assert exe_res == gold_res` inside the program-accuracy branch is
   dropped. It is a debug check, and a single rounding disagreement aborts the
   whole evaluation.

`evaluate_result_official` runs the unmodified protocol for comparison, so the
cost of each correction can be seen rather than assumed.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


# ------------------------------------------------------------------ numbers --
def str_to_num(text: str) -> Union[float, str]:
    """FinQA's literal parser, unchanged: returns "n/a" rather than raising."""
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            # Multiples are written "14.3x" in these tables. Unlike "%", the
            # suffix carries no scaling -- the gold answer for such a row is the
            # plain multiple.
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _cell_to_num(raw: str) -> Union[float, str]:
    """Parse one table cell.

    Adds the `(3344)` form to what the original handled; `$ -1046 ( 1046 )`
    still resolves through the original's "text before the first bracket" rule.
    """
    text = str(raw).replace("$", "").strip()
    m = _PAREN_NEG_RE.match(text)
    if m:
        value = str_to_num(m.group(1))
        return -value if value != "n/a" else "n/a"
    return str_to_num(text.split("(")[0].strip())


_MISSING_CELL_MARKERS = {"", "-", "–", "—", "na", "n/a", "nan", "none"}


def process_row(row_in: Sequence[str]):
    """Numeric values of a table row, or "n/a" if the row cannot be reduced.

    A cell that merely marks a missing period ("-", "NA", an em dash) is
    skipped: rows in this dataset routinely lack a year or two, and voiding the
    whole row over one gap loses reductions the gold answers depend on. A cell
    with real but unreadable content still voids the row, so genuine parse
    failures are not silently averaged away.
    """
    row_out = []
    for cell in row_in:
        text = str(cell).replace("$", "").strip()
        if text.lower() in _MISSING_CELL_MARKERS:
            continue
        num = _cell_to_num(text)
        if num == "n/a":
            return "n/a"
        row_out.append(num)
    return row_out or "n/a"


# -------------------------------------------------------------- tokenisation --
def program_tokenization(original_program: str) -> List[str]:
    """Tokenise into ['op(', arg1, arg2, ')', ..., 'EOF'].

    Bracket-depth aware, so a row label like `ROE (%)` stays one token. The
    original split on every bracket, which shattered such labels and broke the
    four-tokens-per-step structure the rest of the protocol relies on.

    Raises ValueError if trailing, non-whitespace text remains once no further
    step can be parsed (e.g. a step missing its closing paren, which happens
    both in a handful of gold programs and -- more importantly -- in model
    generations cut off by a max_new_tokens limit). An earlier version of this
    tokenizer silently stopped and returned only the steps parsed so far,
    which let a truncated program like "subtract(100, 50), divide(#0, 5"
    (missing text and closing paren) score as a valid, complete one-step
    program instead of being rejected -- a false positive for exactly the kind
    of generation failure this evaluator needs to catch.
    """
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching ')' found) in program: '{original_program}'"
            )

        program.append(m.group(1) + "(")
        # Split arguments on depth-0 commas so brackets inside a label survive.
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: '{text[pos:]}' (from: '{original_program}')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    """Recover a program string from raw model output.

    Bracket matched, so an outer call is never silently discarded: the earlier
    regex could only match a bracket-free call, so `multiply(divide(a, b), 100)`
    was reduced to its inner `divide(a, b)` and a percentage rescaling scored as
    if it were the gold answer.
    """
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text



def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    """Group a tokenised program into (op, arg1, arg2) triples.

    The original walked the token list by joining it and splitting on ")", which
    silently mis-splits any argument containing a bracket -- exactly the row
    labels this dataset uses, e.g. `EPS (VND)`. Grouping the tokens directly is
    equivalent for well-formed programs and correct for those.
    """
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps

# ---------------------------------------------------------------- execution --
def eval_program(program: List[str], table: Optional[Sequence[Sequence[str]]]):
    """Execute a tokenised program. Returns (invalid_flag, result)."""
    this_res: Union[float, str] = "n/a"

    try:
        steps = _steps_from_tokens(program)
        res_dict = {}

        for ind, (op, arg1, arg2) in enumerate(steps):
            if op in ("add", "subtract", "multiply", "divide", "exp", "greater"):
                if "#" in arg1:
                    arg1 = res_dict[int(arg1.replace("#", ""))]
                else:
                    arg1 = str_to_num(arg1)
                    if arg1 == "n/a":
                        return 1, "n/a"
                if "#" in arg2:
                    arg2 = res_dict[int(arg2.replace("#", ""))]
                else:
                    arg2 = str_to_num(arg2)
                    if arg2 == "n/a":
                        return 1, "n/a"

                if op == "add":
                    this_res = arg1 + arg2
                elif op == "subtract":
                    this_res = arg1 - arg2
                elif op == "multiply":
                    this_res = arg1 * arg2
                elif op == "divide":
                    this_res = arg1 / arg2
                elif op == "exp":
                    this_res = arg1 ** arg2
                else:
                    this_res = "yes" if arg1 > arg2 else "no"

            else:  # table_*
                table_dict = {row[0]: row[1:] for row in (table or [])}
                if "#" in arg1:
                    num_row = [res_dict[int(arg1.replace("#", ""))]]
                else:
                    if arg1 not in table_dict:
                        return 1, "n/a"
                    num_row = process_row(table_dict[arg1])
                if num_row == "n/a":
                    return 1, "n/a"

                if op == "table_max":
                    this_res = max(num_row)
                elif op == "table_min":
                    this_res = min(num_row)
                elif op == "table_sum":
                    this_res = sum(num_row)
                else:
                    this_res = sum(num_row) / len(num_row)

            res_dict[ind] = this_res

        if this_res not in ("yes", "no", "n/a"):
            this_res = round(this_res, 5)
    except Exception:
        return 1, "n/a"

    return 0, this_res


# ------------------------------------------------------------------ program --
def equal_program(program1: List[str], program2: List[str]) -> bool:
    """Symbolic equivalence of gold (program1) and prediction (program2).

    Same protocol as the official implementation -- literals become symbols,
    table steps become opaque variables, and the two expressions are compared
    after `simplify`, so a differently-arranged but algebraically identical
    program still counts. A prediction may only use symbols that appear in gold,
    which is what stops it from introducing a constant of its own (the `100` of
    a percentage rescaling, say). Only the step-splitting differs: it groups
    tokens rather than splitting a joined string on ")".
    """
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


# ------------------------------------------------------------------ metrics --
def _coerce_answer(value):
    """ViNumQA stores exe_ans as a string; "yes"/"no" stay as they are."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def score_one(generated_program: str, gold_program: str, gold_answer,
              table: Optional[Sequence[Sequence[str]]] = None,
              extract_first: bool = True) -> Tuple[float, float]:
    """(program_accuracy, execution_accuracy) for a single item.

    A generated_program that fails to tokenize (e.g. cut off mid-generation,
    missing a closing paren) scores (0.0, 0.0) rather than raising -- this is
    expected input from a real model, not a bug to surface as an exception.
    gold_program is assumed well-formed and is not caught the same way, so a
    malformed *gold* label still raises loudly instead of silently scoring 0.
    """
    generated = extract_program(generated_program) if extract_first else generated_program
    gold_tok = program_tokenization(gold_program)
    gold_res = _coerce_answer(gold_answer)

    try:
        pred_tok = program_tokenization(generated)
    except ValueError:
        return 0.0, 0.0

    invalid, exe_res = eval_program(pred_tok, table)
    ea = 1.0 if invalid == 0 and exe_res == gold_res else 0.0

    try:
        pa = 1.0 if equal_program(gold_tok, pred_tok) else 0.0
    except Exception:
        pa = 0.0

    return pa, ea


def evaluate_dataframe(df, generated_col: str = "generated_program",
                       gold_program_col: str = "program",
                       gold_answer_col: str = "answer",
                       table_col: str = "table_raw",
                       extract_first: bool = True):
    """Score a DataFrame, returning (df + per-row scores, summary).

    `table_col` must hold the raw table (list of rows); without it, programs
    naming a table row cannot execute and score 0 on EA.
    """
    df = df.copy()
    pa_scores, ea_scores = [], []

    for _, row in df.iterrows():
        table = row[table_col] if table_col in df.columns else None
        pa, ea = score_one(row[generated_col], row[gold_program_col],
                           row[gold_answer_col], table, extract_first)
        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    return df, {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }


In [ ]:
df_scored, summary = evaluate_dataframe(
    df,
    generated_col="generated_program",
    gold_program_col="program",
    gold_answer_col="answer",
    table_col="table_raw",
)

print("Qwen3-4B-Thinking-2507, 3-shot, baseline (no fine-tuning):")
print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}}


### Save

In [ ]:
results_path = OUTPUT_DIR / "few-shot_qwen3-4b-thinking-2507_results.csv"
summary_path = OUTPUT_DIR / "few-shot_qwen3-4b-thinking-2507_summary.json"

df_scored.to_csv(results_path, index=False)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({"model": "Qwen3-4B-Thinking-2507", "shot": "few-shot", "fine_tuned": False,
              "max_tokens": sampling_params.max_tokens, **summary},
             f, ensure_ascii=False, indent=2)

print(f"Saved per-sample results to {results_path}")
print(f"Saved summary to {summary_path}")
